In [1]:
import pandas as pd 
import numpy as np 

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix


## 1. Data Loading (5 Marks)

In [2]:
df = pd.read_csv("../data/water_potability.csv")
df.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [3]:
df.shape

(3276, 10)

## 2. Data Preprocessing (10 Marks)

### Missing values per column

In [4]:
df.isnull().sum()

ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64

### Replacing Missing Values By Median

In [5]:
df.fillna(df.median(), inplace=True)

### Remove Duplicates

In [6]:
df.drop_duplicates(inplace=True)

### Feature Engineering

In [7]:
df["neutral"] = ((df["ph"] > 6.5) & (df["ph"] < 8.5)).astype(int)
df

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability,neutral
0,7.036752,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0,1
1,3.716080,129.422921,18630.057858,6.635246,333.073546,592.885359,15.180013,56.329076,4.500656,0,0
2,8.099124,224.236259,19909.541732,9.275884,333.073546,418.606213,16.868637,66.420093,3.055934,0,1
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0,1
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0,0
...,...,...,...,...,...,...,...,...,...,...,...
3271,4.668102,193.681735,47580.991603,7.166639,359.948574,526.424171,13.894419,66.687695,4.435821,1,0
3272,7.808856,193.553212,17329.802160,8.061362,333.073546,392.449580,19.903225,66.622485,2.798243,1,1
3273,9.419510,175.762646,33155.578218,7.350233,333.073546,432.044783,11.039070,69.845400,3.298875,1,0
3274,5.126763,230.603758,11983.869376,6.303357,333.073546,402.883113,11.168946,77.488213,4.708658,1,0


### Outlier Detection

In [8]:

features = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

Q1 = df[features].quantile(0.25)
Q3 = df[features].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Boolean mask: True if row has at least one outlier in any feature
outlier_mask = ((df[features] < lower_bound) | (df[features] > upper_bound)).any(axis=1)

print(f"Number of rows with at least one outlier: {outlier_mask.sum()}")


Number of rows with at least one outlier: 610


## 3. Pipeline Creation (10 Marks)

In [9]:
rf_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model',RandomForestClassifier(random_state=42, class_weight='balanced'))
])

## 4. Primary Model Selection (5 Marks)

We choose Random Forest Classifier as it handles non linear relationships well.  
This can handle both numerical and categorical data without extensive preprocessing.  
This algorithm can maintain high accuracy even when a large proportion of data is missing. 

## 5. Model Training (10 Marks)

In [10]:
X = df.drop("Potability", axis=1)
y = df["Potability"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

## 6. Cross-Validation (10 Marks)

In [11]:
cv_scores = cross_val_score( rf_model, X_train, y_train,cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)

avg_score = cv_rmse.mean()
std_dev = cv_rmse.std()

print(f"Avg Score: {avg_score:.4}")
print(f"Std Score: {std_dev:.4}")

Avg Score: 0.586
Std Score: 0.01041


## 7. Hyperparameter Tuning (10 Marks)

In [12]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10],
    "model__min_samples_split": [2, 5],
    "model__max_features": ["sqrt"]
}

grid = GridSearchCV(
    rf_model,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best Score: 0.469700421353084


## 8. Best Model Selection (10 Marks)

In [13]:
best_model = grid.best_estimator_
best_model.fit(X_train, y_train) 

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=10,
                                        min_samples_split=5, n_estimators=200,
                                        random_state=42))])

In [14]:
## 9. Model Performance Evaluation (10 Marks)

In [15]:
y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("")
print("")
print("")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("")
print("")
print("")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6554878048780488



Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.83      0.75       400
           1       0.59      0.38      0.46       256

    accuracy                           0.66       656
   macro avg       0.63      0.61      0.61       656
weighted avg       0.64      0.66      0.64       656




Confusion Matrix:
[[332  68]
 [158  98]]
